In [ ]:
from pathlib import Path
import os 
import pandas as pd 
import os
import tarfile
import urllib.request
from pathlib import Path
from urllib.parse import urlparse


df_train=pd.read_csv('/kaggle/input/kinetics-700-2020-zip-files/train.csv')
set(df_train['label'].to_list())
print('orginal shape is ',df_train.shape)
labels=["push up", "situp", "squat"]
df_train = df_train[df_train["label"].isin(labels)] 
print('current shape is ',df_train.shape)
print(df_train['label'].value_counts())

df_train["filename"] = (
    df_train["youtube_id"].astype(str)
    + "_"
    + df_train["time_start"].astype(int).map(lambda x: f"{x:06d}")
    + "_"
    + df_train["time_end"].astype(int).map(lambda x: f"{x:06d}")
    + ".mp4"
)

df_val=pd.read_csv('/kaggle/input/kinetics-700-2020-zip-files/val.csv')
set(df_val['label'].to_list())
print('orginal shape is ',df_val.shape)
df_val = df_val[df_val["label"].isin(labels)] 
print('current shape is ',df_val.shape)
print(df_val['label'].value_counts())

df_val["filename"] = (
    df_val["youtube_id"].astype(str)
    + "_"
    + df_val["time_start"].astype(int).map(lambda x: f"{x:06d}")
    + "_"
    + df_val["time_end"].astype(int).map(lambda x: f"{x:06d}")
    + ".mp4"
)

print('-'*20)
df=pd.concat([df_val,df_train])
print(df['label'].value_counts())
df.to_csv('df.csv',index=False)
df



s = pd.read_csv('/kaggle/input/kinetics-700-2020-zip-files/train.csv')['label']
unique_list=pd.unique(s).tolist()
labels=["push up", "situp", "squat"]
print('--'*20,'labels')
[print(x,unique_list.index(x)+1) for x in labels]

urls = [
    "https://s3.amazonaws.com/kinetics/700_2020/train/k700_train_461.tar.gz",
    "https://s3.amazonaws.com/kinetics/700_2020/train/k700_train_542.tar.gz",
    "https://s3.amazonaws.com/kinetics/700_2020/train/k700_train_577.tar.gz",
    "https://s3.amazonaws.com/kinetics/700_2020/val/k700_val_461.tar.gz",
    "https://s3.amazonaws.com/kinetics/700_2020/val/k700_val_542.tar.gz",
    "https://s3.amazonaws.com/kinetics/700_2020/val/k700_val_577.tar.gz",
]

train_urls = [u for u in urls if "/k700_train_" in u]
val_urls   = [u for u in urls if "/k700_val_" in u]

out_dir = Path(".")  
out_dir.mkdir(parents=True, exist_ok=True)

train_path = out_dir / "train.txt"
val_path   = out_dir / "val.txt"

train_path.write_text("\n".join(train_urls) + "\n", encoding="utf-8")
val_path.write_text("\n".join(val_urls) + "\n", encoding="utf-8")

print(f"Wrote {len(train_urls)} URLs to {train_path}")
print(f"Wrote {len(val_urls)} URLs to {val_path}")



In [ ]:
from __future__ import annotations

import os
import sys
import tarfile
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence
from urllib.parse import urlparse

import pandas as pd




CHUNK_SIZE = 64 * 1024 * 1024  
KEEP_PREFIXES = ("push", "situp", "squat")




def filename_from_url(url: str) -> str:
    """Best-effort filename extraction from a URL."""
    name = Path(urlparse(url).path).name
    return name if name else "downloaded_file"


def download_file(url: str, dest: Path, chunk_size: int = CHUNK_SIZE) -> Path:
    """
    Download a file to `dest` with a simple progress indicator.

    Skips download if `dest` already exists.
    """
    dest.parent.mkdir(parents=True, exist_ok=True)

    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] already downloaded: {dest}")
        return dest

    print(f"[download] {url}")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

    with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length") or 0)
        downloaded = 0

        while True:
            chunk = r.read(chunk_size)
            if not chunk:
                break
            f.write(chunk)
            downloaded += len(chunk)

            if total:
                pct = (downloaded * 100.0) / total
                sys.stdout.write(
                    f"\r  {dest.name}: {downloaded/1e6:.1f}/{total/1e6:.1f} MB ({pct:.1f}%)"
                )
                sys.stdout.flush()

        if total:
            print()  

    return dest


def _is_within_directory(base_dir: Path, target: Path) -> bool:
    """Return True if `target` resolves under `base_dir`."""
    base = base_dir.resolve()
    tgt = target.resolve()
    return os.path.commonpath([str(base)]) == os.path.commonpath([str(base), str(tgt)])


def safe_extract(tar: tarfile.TarFile, out_dir: Path) -> None:
    """
    Safely extract tar to `out_dir`:
    - blocks path traversal
    - blocks symlinks/hardlinks (common tar escape vector)
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    for member in tar.getmembers():
        if member.issym() or member.islnk():
            raise RuntimeError(f"Blocked link entry in tar: {member.name}")

        member_path = out_dir / member.name
        if not _is_within_directory(out_dir, member_path):
            raise RuntimeError(f"Blocked suspicious path in tar: {member.name}")

    tar.extractall(out_dir)


def extract_if_tar(archive_path: Path, out_dir: Path) -> bool:
    """Extract .tar / .tar.gz / .tgz archives. Returns True if extracted."""
    name = archive_path.name.lower()
    if not (name.endswith(".tar") or name.endswith(".tar.gz") or name.endswith(".tgz")):
        print(f"[skip] not a tar archive: {archive_path.name}")
        return False

    print(f"[extract] {archive_path.name} -> {out_dir}")
    with tarfile.open(archive_path, "r:*") as tar:
        safe_extract(tar, out_dir)
    return True


def prune_split_dir(split_dir: Path, keep_prefixes: Sequence[str] = KEEP_PREFIXES) -> None:
    """
    Remove top-level items in `split_dir` except those whose names start with `keep_prefixes`.
    Mirrors your `find ... ! -name 'push*' ! -name 'situp*' ! -name 'squat*' -exec rm -rf` logic,
    but purely in Python.
    """
    keep_prefixes = tuple(keep_prefixes)

    to_delete: list[Path] = []
    for p in split_dir.iterdir():
        if p.name.startswith(keep_prefixes):
            continue
        to_delete.append(p)

    if not to_delete:
        return

    print("[prune] delete the below items .......")
    for p in to_delete:
        print(f"  - {p}")

    for p in to_delete:
        if p.is_dir():
            
            for child in sorted(p.rglob("*"), reverse=True):
                try:
                    child.unlink() if child.is_file() or child.is_symlink() else child.rmdir()
                except FileNotFoundError:
                    pass
            try:
                p.rmdir()
            except FileNotFoundError:
                pass
        else:
            try:
                p.unlink()
            except FileNotFoundError:
                pass

    print("-" * 20)




@dataclass(frozen=True)
class SplitJob:
    split: str
    url_list_path: Path


def process_urls(urls: Iterable[str], out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    for url in urls:
        url = str(url).strip()
        if not url:
            continue

        archive_path = out_dir / filename_from_url(url)

        try:
            downloaded = download_file(url, archive_path)
            extract_if_tar(downloaded, out_dir)
            prune_split_dir(out_dir, KEEP_PREFIXES)

        except Exception as e:
            print(f"[error] {url}: {e}")


def load_urls_from_txt_csv(path: Path) -> list[str]:
    """Your Kaggle file is a 1-column CSV; load it cleanly."""
    return pd.read_csv(path, header=None)[0].astype(str).tolist()


def run_all(jobs: Sequence[SplitJob]) -> None:
    for job in jobs:
        print(f"\n=== split: {job.split} ===")
        urls = load_urls_from_txt_csv(job.url_list_path)
        process_urls(urls, out_dir=Path(job.split))




jobs = [
    SplitJob("train", Path("train.txt")),
    
    SplitJob("val",   Path("val.txt")),
]

run_all(jobs)

In [ ]:
















import os
import sys
import re
import json
import time
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

WORK_ROOT = Path("/kaggle/working")

VIDEO_ROOTS = {
    "train": WORK_ROOT / "train",
    "val": WORK_ROOT / "val",
}

FLAT_VIDEO_DIRS = {
    "train": WORK_ROOT / "train_flat",
    "val": WORK_ROOT / "val_flat",
    "test": WORK_ROOT / "test_flat",
}

CSV_PATHS = {
    "train": Path("/kaggle/input/kinetics-700-2020-zip-files/train.csv"),
    "val":   Path("/kaggle/input/kinetics-700-2020-zip-files/val.csv"),
}

CLASSES = ["push up", "situp", "squat"]


TSN_K400_CKPT = (
    "https://download.openmmlab.com/mmaction/v1.0/recognition/tsn/"
    "tsn_imagenet-pretrained-r50_8xb32-1x1x8-100e_kinetics400-rgb/"
    "tsn_imagenet-pretrained-r50_8xb32-1x1x8-100e_kinetics400-rgb_20220906-2692d16c.pth"
)

N_TRAIN_PER_CLASS = 700
N_VAL_PER_CLASS = 100
N_TEST_PER_CLASS = 100
RANDOM_SEED = 42



def run(cmd, cwd=None, extra_env=None):
    print("\n[RUN]", " ".join(map(str, cmd)))
    env = None
    if extra_env:
        env = {**os.environ, **extra_env}
    subprocess.check_call(list(map(str, cmd)), cwd=cwd, env=env)


def pip_install(pkgs):
    run([sys.executable, "-m", "pip", "install", "-U", *pkgs])


def pip_uninstall(pkgs):
    run([sys.executable, "-m", "pip", "uninstall", "-y", *pkgs])


def ensure_video_roots():
    for split, d in VIDEO_ROOTS.items():
        if not d.exists() or not d.is_dir():
            raise FileNotFoundError(f"Missing required folder: {d}")
    print("[DATA] Video roots found:")
    for split, d in VIDEO_ROOTS.items():
        mp4s = list(d.rglob("*.mp4"))
        print(f"  - {split}: {d} (mp4 count: {len(mp4s)})")
        if len(mp4s) == 0:
            raise RuntimeError(f"No .mp4 under {d}.")
        for ex in mp4s[:3]:
            print(f"      example: {ex.relative_to(d).as_posix()}")


def patch_opencv_typing_if_needed():
    try:
        import cv2
        cv2_dir = Path(cv2.__file__).resolve().parent
        p = cv2_dir / "typing" / "__init__.py"
        if p.exists():
            txt = p.read_text(encoding="utf-8", errors="ignore")
            bad = "LayerId = cv2.dnn.DictValue"
            if bad in txt and f"#{bad}" not in txt:
                p.write_text(txt.replace(bad, f"#{bad}"), encoding="utf-8")
                print(f"[PATCH] Patched OpenCV typing stub: {p}")
    except Exception as e:
        print(f"[PATCH] Skipped OpenCV patch ({e})")



def patch_mmaction2_disable_multimodal(repo_dir: Path):
    p = repo_dir / "mmaction" / "models" / "__init__.py"
    if not p.exists():
        return
    txt = p.read_text(encoding="utf-8")
    target = "from .multimodal import *  # noqa: F401,F403"
    if target in txt and "try:\n    from .multimodal import *" not in txt:
        txt = txt.replace(
            target,
            "try:\n"
            "    from .multimodal import *  # noqa: F401,F403\n"
            "except Exception:\n"
            "    pass\n",
        )
        p.write_text(txt, encoding="utf-8")
        print("[PATCH] Disabled multimodal auto-import.")


def install_mmaction2():
    pip_install([
        "pip>=23.3", "setuptools>=70.0.0", "wheel>=0.42.0",
        "packaging>=23.2", "tqdm", "mmaction2",
    ])
    patch_opencv_typing_if_needed()

    pip_install(["scikit-learn", "matplotlib"])
    pip_uninstall(["mmcv", "mmcv-lite"])
    pip_install(["mmengine==0.10.7"])
    pip_install(["mmcv-lite==2.1.0"])
    pip_install(["decord"])

    repo_dir = Path("mmaction2").resolve()
    if not repo_dir.exists():
        run(["git", "clone", "--depth", "1",
             "https://github.com/open-mmlab/mmaction2.git"])

    patch_mmaction2_disable_multimodal(repo_dir)
    run([sys.executable, "-m", "pip", "install", "-v", "-e", "."],
        cwd=str(repo_dir))
    return repo_dir



def parse_video_keys(video_root: Path):
    key_to_path = {}
    for p in sorted(video_root.rglob("*.mp4")):
        stem = p.stem
        parts = stem.rsplit("_", 2)
        if len(parts) != 3:
            continue
        vid, s, e = parts
        if not (s.isdigit() and e.isdigit()):
            continue
        key = (vid, int(s), int(e))
        key_to_path.setdefault(key, p)
    return key_to_path


def safe_link_or_copy(src: Path, dst: Path):
    try:
        if dst.exists() or dst.is_symlink():
            return
        dst.symlink_to(src.resolve())
    except Exception:
        shutil.copy2(src, dst)



def load_filtered_df(csv_path: Path, split_name: str):
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    required = {"youtube_id", "time_start", "time_end", "label"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"{split_name}.csv missing columns: {sorted(missing)}")
    df = df[df["label"].isin(CLASSES)].copy()
    df["youtube_id"] = df["youtube_id"].astype(str)
    df["time_start"] = df["time_start"].astype(int)
    df["time_end"] = df["time_end"].astype(int)
    df["key"] = list(zip(df["youtube_id"], df["time_start"], df["time_end"]))
    return df


def keep_only_existing(df: pd.DataFrame, key_to_path: dict, split_name: str):
    before = len(df)
    df2 = df[df["key"].isin(key_to_path.keys())].copy()
    after = len(df2)
    print(f"[DATA] {split_name}: kept {after}/{before} rows with existing mp4")
    if after == 0:
        raise RuntimeError(f"[ERROR] {split_name}: 0 rows match mp4 files.")
    return df2


def sample_per_label(df: pd.DataFrame, n_per_label: int, seed: int, tag: str):
    out = []
    for lab in CLASSES:
        sub = df[df["label"] == lab]
        if len(sub) == 0:
            print(f"[WARN] {tag}: label '{lab}' has 0 samples.")
            continue
        if len(sub) < n_per_label:
            print(f"[WARN] {tag}: label '{lab}' has {len(sub)} < {n_per_label}; taking all.")
            out.append(sub)
        else:
            out.append(sub.sample(n=n_per_label, random_state=seed))
    if not out:
        return df.iloc[0:0].copy()
    return pd.concat(out, ignore_index=True)


def build_splits(df_train: pd.DataFrame, df_val: pd.DataFrame):
    combined = pd.concat(
        [df_train.assign(_src="train"), df_val.assign(_src="val")],
        ignore_index=True,
    )

    
    df_test = sample_per_label(combined, N_TEST_PER_CLASS, RANDOM_SEED, "TEST")
    test_keys = set(df_test["key"].tolist())

    df_train_rem = df_train[~df_train["key"].isin(test_keys)].copy()
    df_val_rem = df_val[~df_val["key"].isin(test_keys)].copy()

    
    df_train_sel = sample_per_label(
        df_train_rem, N_TRAIN_PER_CLASS, RANDOM_SEED + 1, "TRAIN(train_pool)")
    train_keys = set(df_train_sel["key"].tolist())

    
    need_topup = {
        lab: max(0, N_TRAIN_PER_CLASS - int((df_train_sel["label"] == lab).sum()))
        for lab in CLASSES
    }
    if any(v > 0 for v in need_topup.values()):
        pool = df_val_rem[~df_val_rem["key"].isin(train_keys)].copy()
        top_parts = []
        for lab, need in need_topup.items():
            if need <= 0:
                continue
            sub = pool[pool["label"] == lab]
            take_n = min(need, len(sub))
            if take_n < need:
                print(f"[WARN] TRAIN top-up: '{lab}' short by {need - take_n}.")
            if take_n > 0:
                top_parts.append(sub.sample(n=take_n, random_state=RANDOM_SEED + 10))
        if top_parts:
            df_train_sel = pd.concat([df_train_sel, *top_parts], ignore_index=True)
            train_keys = set(df_train_sel["key"].tolist())

    
    df_val_pool2 = df_val_rem[~df_val_rem["key"].isin(train_keys)].copy()
    df_val_sel = sample_per_label(
        df_val_pool2, N_VAL_PER_CLASS, RANDOM_SEED + 2, "VAL(val_pool)")
    val_keys = set(df_val_sel["key"].tolist())

    
    need_topup_val = {
        lab: max(0, N_VAL_PER_CLASS - int((df_val_sel["label"] == lab).sum()))
        for lab in CLASSES
    }
    if any(v > 0 for v in need_topup_val.values()):
        pool = df_train_rem[~df_train_rem["key"].isin(train_keys | val_keys)].copy()
        top_parts = []
        for lab, need in need_topup_val.items():
            if need <= 0:
                continue
            sub = pool[pool["label"] == lab]
            take_n = min(need, len(sub))
            if take_n < need:
                print(f"[WARN] VAL top-up: '{lab}' short by {need - take_n}.")
            if take_n > 0:
                top_parts.append(sub.sample(n=take_n, random_state=RANDOM_SEED + 20))
        if top_parts:
            df_val_sel = pd.concat([df_val_sel, *top_parts], ignore_index=True)
            val_keys = set(df_val_sel["key"].tolist())

    
    train_keys = set(df_train_sel["key"].tolist())
    val_keys = set(df_val_sel["key"].tolist())
    test_keys = set(df_test["key"].tolist())
    if (train_keys & val_keys) or (train_keys & test_keys) or (val_keys & test_keys):
        raise RuntimeError("[ERROR] Split overlap detected.")

    def report(df_, name):
        counts = {lab: int((df_["label"] == lab).sum()) for lab in CLASSES}
        print(f"[SPLIT] {name} counts:", counts, " total=", len(df_))

    report(df_train_sel, "TRAIN")
    report(df_val_sel, "VAL")
    report(df_test, "TEST")
    return df_train_sel, df_val_sel, df_test



def flatten_and_write_split(
    df_split: pd.DataFrame, split_name: str,
    key_to_path_all: dict, label_to_idx: dict,
):
    flat_dir = FLAT_VIDEO_DIRS[split_name]
    flat_dir.mkdir(parents=True, exist_ok=True)

    ann_root = WORK_ROOT / "mmaction2_custom_data" / "annotations"
    ann_root.mkdir(parents=True, exist_ok=True)
    ann_path = ann_root / f"{split_name}.txt"

    lines = []
    used_names = set()
    for _, r in df_split.iterrows():
        key = r["key"]
        src = key_to_path_all.get(key)
        if src is None:
            continue
        vid, s, e = key
        safe_vid = str(vid).replace(" ", "_")
        out_name = f"{safe_vid}_{int(s):06d}_{int(e):06d}.mp4"
        if out_name in used_names:
            out_name = f"{safe_vid}_{int(s):06d}_{int(e):06d}_{len(used_names)}.mp4"
        used_names.add(out_name)
        safe_link_or_copy(src, flat_dir / out_name)
        y = label_to_idx[str(r["label"])]
        lines.append(f"{out_name} {y}")

    ann_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    print(f"[ANN] {split_name}: wrote {len(lines)} lines -> {ann_path}")
    if len(lines) == 0:
        raise RuntimeError(f"[ERROR] {split_name} annotation is empty.")
    return str(ann_path), str(flat_dir.resolve())


def write_all_annotations_and_flatten(df_train, df_val, df_test, key_to_path_all):
    ann_root = WORK_ROOT / "mmaction2_custom_data" / "annotations"
    ann_root.mkdir(parents=True, exist_ok=True)
    label_to_idx = {c: i for i, c in enumerate(CLASSES)}
    (ann_root / "classes.txt").write_text("\n".join(CLASSES) + "\n", encoding="utf-8")
    (ann_root / "label_to_idx.json").write_text(
        json.dumps(label_to_idx, indent=2), encoding="utf-8")

    ann_train, train_pfx = flatten_and_write_split(
        df_train, "train", key_to_path_all, label_to_idx)
    ann_val, val_pfx = flatten_and_write_split(
        df_val, "val", key_to_path_all, label_to_idx)
    ann_test, test_pfx = flatten_and_write_split(
        df_test, "test", key_to_path_all, label_to_idx)

    ann_files = {"train": ann_train, "val": ann_val, "test": ann_test}
    prefixes = {"train": train_pfx, "val": val_pfx, "test": test_pfx}
    return ann_files, prefixes, label_to_idx



def choose_base_tsn_config(repo_dir: Path) -> Path:
    """Pick the TSN config that matches our checkpoint architecture."""
    tsn_dir = repo_dir / "configs" / "recognition" / "tsn"
    if not tsn_dir.exists():
        raise FileNotFoundError(f"TSN config dir not found: {tsn_dir}")

    
    cands = sorted(tsn_dir.glob("*imagenet*r50*kinetics400*rgb*.py"))
    if cands:
        return cands[0]

    
    cands = sorted(tsn_dir.glob("*kinetics400*.py"))
    if cands:
        return cands[0]

    
    cands = sorted(tsn_dir.glob("*.py"))
    if not cands:
        raise FileNotFoundError("No TSN configs found.")
    return cands[0]


def _get_inner_dataset(ds):
    if isinstance(ds, dict) and "dataset" in ds:
        return ds["dataset"]
    return ds


def write_custom_config(repo_dir: Path, ann_files, prefixes):
    from mmengine.config import Config

    base_cfg_path = choose_base_tsn_config(repo_dir)
    cfg = Config.fromfile(str(base_cfg_path))

    work_dir = WORK_ROOT / "work_dirs" / "tsn_k400_3class"
    work_dir.mkdir(parents=True, exist_ok=True)
    cfg.work_dir = str(work_dir.resolve())

    
    model = cfg.get("model", None)
    if model is None:
        raise KeyError("Config has no 'model' key.")

    if "cls_head" in model:
        model["cls_head"]["num_classes"] = len(CLASSES)
    elif "head" in model:
        model["head"]["num_classes"] = len(CLASSES)
    else:
        raise KeyError("Could not locate model head to set num_classes.")

    
    
    
    
    
    
    if "backbone" in model and isinstance(model["backbone"], dict):
        model["backbone"].pop("pretrained", None)
        model["backbone"].pop("init_cfg", None)

    cfg.model = model
    cfg.load_from = TSN_K400_CKPT

    
    def set_loader(loader_key, split, test_mode):
        loader = cfg.get(loader_key, None)
        if loader is None:
            return
        ds = loader.get("dataset", None)
        if ds is None:
            return
        inner = _get_inner_dataset(ds)
        inner["ann_file"] = ann_files[split]
        inner["data_prefix"] = dict(video=prefixes[split])
        inner["test_mode"] = bool(test_mode)
        inner["metainfo"] = dict(classes=CLASSES)
        inner["serialize_data"] = False
        if isinstance(ds, dict) and "dataset" in ds:
            ds["dataset"] = inner
            loader["dataset"] = ds
        else:
            loader["dataset"] = inner
        cfg[loader_key] = loader

    set_loader("train_dataloader", "train", test_mode=False)
    set_loader("val_dataloader", "val", test_mode=True)
    set_loader("test_dataloader", "test", test_mode=True)

    
    if "train_dataloader" in cfg:
        cfg.train_dataloader["batch_size"] = 16
        cfg.train_dataloader["num_workers"] = 4
    if "val_dataloader" in cfg:
        cfg.val_dataloader["batch_size"] = 16
        cfg.val_dataloader["num_workers"] = 4
    if "test_dataloader" in cfg:
        cfg.test_dataloader["batch_size"] = 16
        cfg.test_dataloader["num_workers"] = 4

    
    max_epochs = 30

    cfg.train_cfg = dict(
        type="EpochBasedTrainLoop", max_epochs=max_epochs, val_interval=1)
    cfg.val_cfg = dict(type="ValLoop")
    cfg.test_cfg = dict(type="TestLoop")

    
    
    
    
    cfg.optim_wrapper = dict(
        optimizer=dict(type="SGD", lr=0.005, momentum=0.9, weight_decay=1e-4),
        clip_grad=dict(max_norm=40, norm_type=2),
    )

    
    
    
    warmup_epochs = 3
    cfg.param_scheduler = [
        dict(
            type="LinearLR",
            start_factor=0.1,
            by_epoch=True,
            begin=0,
            end=warmup_epochs,
        ),
        dict(
            type="CosineAnnealingLR",
            T_max=max_epochs - warmup_epochs,
            eta_min=1e-5,
            by_epoch=True,
            begin=warmup_epochs,
            end=max_epochs,
        ),
    ]

    
    if "default_hooks" in cfg:
        ck = cfg.default_hooks.get("checkpoint", {})
        ck["interval"] = 1
        ck["max_keep_ckpts"] = 3
        ck["save_best"] = "auto"
        cfg.default_hooks["checkpoint"] = ck

    
    cfg.val_evaluator = dict(type="AccMetric")
    cfg.test_evaluator = dict(type="AccMetric")

    out_cfg = repo_dir / "configs" / "custom" / "tsn_k400_3class.py"
    out_cfg.parent.mkdir(parents=True, exist_ok=True)
    cfg.dump(str(out_cfg))

    print(f"[CFG] Base config:    {base_cfg_path}")
    print(f"[CFG] Custom config:  {out_cfg}")
    print(f"[CFG] Work dir:       {work_dir}")
    print(f"[CFG] Pretrained:     {TSN_K400_CKPT}")
    print(f"[CFG] Batch size:     16")
    print(f"[CFG] LR:             0.005 (SGD + cosine annealing)")
    print(f"[CFG] Epochs:         {max_epochs}")
    return out_cfg, work_dir



def find_best_checkpoint(work_dir: Path) -> Path:
    best = sorted(work_dir.glob("best_*.pth"), key=lambda p: p.stat().st_mtime)
    if best:
        return best[-1]
    epoch = sorted(work_dir.glob("epoch_*.pth"), key=lambda p: p.stat().st_mtime)
    if epoch:
        return epoch[-1]
    raise FileNotFoundError(f"No checkpoints found in {work_dir}")


def parse_mmengine_log_for_history(work_dir: Path):
    
    logs = sorted(work_dir.rglob("*.log"), key=lambda p: p.stat().st_mtime)
    if not logs:
        return {}

    log_path = logs[-1]
    print(f"[LOG] Parsing history from: {log_path}")
    hist = {}
    
    epoch_re = re.compile(r"Epoch\((train|val|test)\)\s*\[(\d+)\]")
    kv_re = re.compile(r"([A-Za-z0-9_\/\.\-]+):\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")

    with log_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            m = epoch_re.search(line)
            if not m:
                continue
            phase = m.group(1)
            epoch = int(m.group(2))
            for k, v in kv_re.findall(line):
                try:
                    val = float(v)
                except Exception:
                    continue
                key = f"{phase}.{k}"
                hist.setdefault(key, []).append((epoch, val))

    out = {}
    for k, pairs in hist.items():
        pairs = sorted(pairs, key=lambda t: t[0])
        xs = np.array([p[0] for p in pairs], dtype=int)
        ys = np.array([p[1] for p in pairs], dtype=float)
        out[k] = (xs, ys)
    return out


def plot_history(history: dict, work_dir: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    if not history:
        print("[PLOT] No history found to plot.")
        return

    candidates = [
        "train.loss", "train.loss_cls", "train.lr",
        "val.acc", "val.top1_acc", "val.accuracy", "val.acc_top1",
    ]
    keys_to_plot = [k for k in candidates if k in history]
    if not keys_to_plot:
        keys_to_plot = list(history.keys())[:6]

    for k in keys_to_plot:
        xs, ys = history[k]
        plt.figure()
        plt.plot(xs, ys, marker="o", markersize=3)
        plt.title(k)
        plt.xlabel("epoch")
        plt.ylabel(k.split(".", 1)[-1])
        plt.grid(True, alpha=0.3)
        out = work_dir / f"history_{k.replace('/','_').replace('.','_')}.png"
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.close()
        print(f"[PLOT] saved {out}")

def summarize_latency(latencies):
    """
    latencies: list/array of seconds per video
    """

    latencies = np.asarray(latencies, dtype=float)

    return {
        "n": int(len(latencies)),

        "mean_s": float(np.mean(latencies)),
        "std_s": float(np.std(latencies, ddof=1)) if len(latencies) > 1 else 0.0,
        "median_s": float(np.median(latencies)),
        "p95_s": float(np.percentile(latencies, 95)),
        "min_s": float(np.min(latencies)),
        "max_s": float(np.max(latencies)),

        "mean_ms": float(np.mean(latencies) * 1000),
        "std_ms": float(
            np.std(latencies, ddof=1) * 1000
            if len(latencies) > 1 else 0.0
        ),
        "median_ms": float(np.median(latencies) * 1000),
        "p95_ms": float(np.percentile(latencies, 95) * 1000),
        "min_ms": float(np.min(latencies) * 1000),
        "max_ms": float(np.max(latencies) * 1000),
    }

def evaluate_on_test(
    cfg_path, best_ckpt, test_ann, test_prefix, work_dir,
    n_warmup=10,
    warmup_video=None,
):
    cfg_path, best_ckpt = Path(cfg_path), Path(best_ckpt)
    test_ann, test_prefix, work_dir = Path(test_ann), Path(test_prefix), Path(work_dir)

    from mmengine.config import Config
    import torch
    from mmaction.apis import init_recognizer, inference_recognizer
    from sklearn.metrics import (
        confusion_matrix, classification_report,
        accuracy_score, balanced_accuracy_score,
        precision_recall_fscore_support, f1_score,
        cohen_kappa_score, matthews_corrcoef,
        log_loss, top_k_accuracy_score,
    )
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    
    
    import mmengine.runner.checkpoint as _ckpt_mod
    _orig_torch_load = torch.load

    def _patched_torch_load(f, map_location=None, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _orig_torch_load(f, map_location=map_location, **kwargs)

    _ckpt_mod.torch.load = _patched_torch_load

    cfg = Config.fromfile(str(cfg_path))
    device = "cuda:0" if torch.cuda.is_available() else "cpu"

    print(f"[LATENCY] device: {device}")
    if torch.cuda.is_available():
        print(f"[LATENCY] GPU: {torch.cuda.get_device_name(0)}")

    model = init_recognizer(cfg, checkpoint=str(best_ckpt), device=device)
    model.eval()

    items = []
    with open(test_ann, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            fn, y = line.split()
            items.append((fn, int(y)))

    if not items:
        raise RuntimeError("Test annotation is empty.")

    
    
    
    
    
    actual_warmup = int(n_warmup)

    if warmup_video is None:
        
        
        warmup_video = test_prefix / items[0][0]
        print("[LATENCY] WARNING: warm-up is using a test video fallback.")
    else:
        warmup_video = Path(warmup_video)

    if not warmup_video.exists():
        raise FileNotFoundError(f"Warm-up video not found: {warmup_video}")

    print(f"[LATENCY] warm-up calls: {actual_warmup}")
    print(f"[LATENCY] warm-up video: {warmup_video}")

    for _ in range(actual_warmup):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        with torch.inference_mode():
            _ = inference_recognizer(model, str(warmup_video))
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    
    
    
    
    
    y_true, y_pred, y_scores, rows = [], [], [], []
    latencies_s = []

    for idx, (fn, gt) in enumerate(items, 1):
        video_path = test_prefix / fn

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        with torch.inference_mode():
            res = inference_recognizer(model, str(video_path))

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        latency_s = time.perf_counter() - t0
        latencies_s.append(latency_s)

        score = getattr(res, "pred_score", None)
        if score is None:
            score = getattr(res, "pred_scores", None)
        if score is None:
            raise RuntimeError("Could not read pred_score from inference result.")

        if hasattr(score, "detach"):
            score = score.detach().cpu().numpy()
        else:
            score = np.asarray(score)

        score = score.reshape(-1)
        pred = int(score.argmax())

        y_true.append(gt)
        y_pred.append(pred)
        y_scores.append(score)

        vid_id = fn.rsplit(".", 1)[0]
        gt_name = CLASSES[gt] if 0 <= gt < len(CLASSES) else str(gt)
        pred_name = CLASSES[pred] if 0 <= pred < len(CLASSES) else str(pred)

        print(
            f"[{idx:03d}/{len(items):03d}] "
            f"{vid_id} | GT={gt_name} | Pred={pred_name} | "
            f"{latency_s * 1000.0:.2f} ms"
        )

        rows.append({
            "video": fn,
            "video_id": vid_id,
            "gt": gt,
            "gt_name": gt_name,
            "pred": pred,
            "pred_name": pred_name,
            "correct": int(pred == gt),
            "p_pred": float(score[pred]),
            "p_gt": float(score[gt]) if 0 <= gt < len(score) else None,
            "scores": json.dumps(score.tolist()),
            "latency_s": float(latency_s),
            "latency_ms": float(latency_s * 1000.0),
        })

    latency_summary = summarize_latency(latencies_s)

    print("\n" + "=" * 60)
    print("MMACTION2 / TSN LATENCY SUMMARY")
    print("=" * 60)
    print(f"N       : {latency_summary['n']}")
    print(f"Mean    : {latency_summary['mean_ms']:.2f} ms/video")
    print(f"SD      : {latency_summary['std_ms']:.2f} ms")
    print(f"Median  : {latency_summary['median_ms']:.2f} ms/video")
    print(f"P95     : {latency_summary['p95_ms']:.2f} ms/video")
    print(f"Min     : {latency_summary['min_ms']:.2f} ms/video")
    print(f"Max     : {latency_summary['max_ms']:.2f} ms/video")

    
    
    
    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    y_scores = np.stack(y_scores, axis=0)

    acc = float(accuracy_score(y_true, y_pred))
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    f1_macro = float(f1_score(y_true, y_pred, average="macro"))
    f1_micro = float(f1_score(y_true, y_pred, average="micro"))
    f1_weighted = float(f1_score(y_true, y_pred, average="weighted"))

    prec_m, rec_m, _, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    prec_w, rec_w, _, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    kappa = float(cohen_kappa_score(y_true, y_pred))
    mcc = float(matthews_corrcoef(y_true, y_pred))

    try:
        ll = float(log_loss(
            y_true, y_scores,
            labels=list(range(len(CLASSES)))
        ))
    except Exception:
        ll = None

    try:
        top1 = float(top_k_accuracy_score(
            y_true, y_scores, k=1,
            labels=list(range(len(CLASSES)))
        ))
        top2 = float(top_k_accuracy_score(
            y_true, y_scores, k=min(2, len(CLASSES)),
            labels=list(range(len(CLASSES)))
        ))
    except Exception:
        top1, top2 = None, None

    cm = confusion_matrix(
        y_true, y_pred,
        labels=list(range(len(CLASSES)))
    )
    cm_norm = cm.astype(float) / np.maximum(
        1.0, cm.sum(axis=1, keepdims=True)
    )

    report = classification_report(
        y_true, y_pred,
        target_names=CLASSES,
        digits=4,
        zero_division=0
    )

    metrics = {
        "n_test": int(len(items)),
        "acc": acc,
        "balanced_acc": bal_acc,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "f1_weighted": f1_weighted,
        "precision_macro": float(prec_m),
        "recall_macro": float(rec_m),
        "precision_weighted": float(prec_w),
        "recall_weighted": float(rec_w),
        "cohen_kappa": kappa,
        "mcc": mcc,
        "log_loss": ll,
        "top1": top1,
        "top2": top2,
        "latency": latency_summary,
        "latency_scope": (
            "Complete mmaction.apis.inference_recognizer call, "
            "including its internal video decoding/sampling/transforms "
            "and model inference; CUDA synchronized before and after timing."
        ),
        "warmup_calls": actual_warmup,
        "device": device,
        "gpu_name": torch.cuda.get_device_name(0)
                    if torch.cuda.is_available() else None,
    }

    print("\n[TEST] classification_report:\n", report)
    print("[TEST] metrics:", json.dumps(metrics, indent=2))

    
    
    
    df_out = pd.DataFrame(rows)

    pred_csv = work_dir / "test_predictions_with_latency.csv"
    df_out.to_csv(pred_csv, index=False)
    print(f"[SAVE] {pred_csv}")

    latency_csv = work_dir / "mmaction2_latency_per_video.csv"
    df_out[
        ["video", "gt_name", "pred_name", "correct",
         "latency_s", "latency_ms"]
    ].to_csv(latency_csv, index=False)
    print(f"[SAVE] {latency_csv}")

    latency_json = work_dir / "mmaction2_latency_summary.json"
    latency_record = dict(latency_summary)
    latency_record.update({
        "timing_scope": (
            "Complete mmaction.apis.inference_recognizer call, including "
            "internal video decoding/sampling/transforms and model inference."
        ),
        "warmup_calls": actual_warmup,
        "warmup_video_source": "training_set",
        "device": device,
        "gpu_name": torch.cuda.get_device_name(0)
                    if torch.cuda.is_available() else None,
    })
    latency_json.write_text(
        json.dumps(latency_record, indent=2),
        encoding="utf-8"
    )
    print(f"[SAVE] {latency_json}")

    metrics_json = work_dir / "test_metrics.json"
    metrics_json.write_text(
        json.dumps(metrics, indent=2),
        encoding="utf-8"
    )
    print(f"[SAVE] {metrics_json}")

    report_txt = work_dir / "classification_report.txt"
    report_txt.write_text(report, encoding="utf-8")
    print(f"[SAVE] {report_txt}")

    
    
    
    def plot_cm(mat, title, out_path, normalize=False):
        plt.figure(figsize=(6, 5))
        plt.imshow(mat, interpolation="nearest", cmap="Blues")
        plt.title(title)
        plt.colorbar()
        tick_marks = np.arange(len(CLASSES))
        plt.xticks(tick_marks, CLASSES, rotation=45, ha="right")
        plt.yticks(tick_marks, CLASSES)

        fmt = ".2f" if normalize else "d"
        thresh = mat.max() / 2.0 if mat.size else 0

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                plt.text(
                    j, i, format(mat[i, j], fmt),
                    ha="center", va="center",
                    color="white" if mat[i, j] > thresh else "black"
                )

        plt.ylabel("GT")
        plt.xlabel("Pred")
        plt.tight_layout()
        plt.savefig(out_path, dpi=180)
        plt.close()
        print(f"[PLOT] saved {out_path}")

    plot_cm(
        cm, "Confusion Matrix (counts)",
        work_dir / "confusion_matrix.png",
        normalize=False
    )
    plot_cm(
        cm_norm, "Confusion Matrix (normalized)",
        work_dir / "confusion_matrix_norm.png",
        normalize=True
    )

    return metrics

def extract_research_details(
    cfg_path: Path, work_dir: Path,
    df_train, df_val, df_test, best_ckpt: Path,
):
    from mmengine.config import Config
    cfg = Config.fromfile(str(cfg_path))
    details = {
        "date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "classes": CLASSES,
        "sampling": {
            "train_per_class": N_TRAIN_PER_CLASS,
            "val_per_class": N_VAL_PER_CLASS,
            "test_per_class": N_TEST_PER_CLASS,
            "seed": RANDOM_SEED,
        },
        "split_sizes": {
            "train": int(len(df_train)),
            "val": int(len(df_val)),
            "test": int(len(df_test)),
        },
        "class_counts": {
            "train": {c: int((df_train["label"] == c).sum()) for c in CLASSES},
            "val": {c: int((df_val["label"] == c).sum()) for c in CLASSES},
            "test": {c: int((df_test["label"] == c).sum()) for c in CLASSES},
        },
        "model_type": str(cfg.get("model", {}).get("type", "")),
        "backbone": cfg.get("model", {}).get("backbone", {}),
        "head": cfg.get("model", {}).get("cls_head",
                                          cfg.get("model", {}).get("head", {})),
        "optimizer": cfg.get("optim_wrapper", {}),
        "scheduler": cfg.get("param_scheduler", None),
        "max_epochs": cfg.get("train_cfg", {}).get("max_epochs", None),
        "best_checkpoint": str(best_ckpt),
        "work_dir": str(work_dir),
        "pretrained_ckpt": TSN_K400_CKPT,
    }
    out = work_dir / "research_details.json"
    out.write_text(json.dumps(details, indent=2, default=str), encoding="utf-8")
    print(f"[SAVE] {out}")
    return details




def save_persistent_artifacts(
    cfg_path: Path,
    work_dir: Path,
    best_ckpt: Path,
    ann_files: dict,
):
    """
    Copy the small, reproducibility-critical outputs to one directory and zip it.
    Kaggle will persist files under /kaggle/working after Save Version.
    """
    artifact_dir = WORK_ROOT / "fitness_paper_mmaction2_artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)

    def copy_if_exists(src, dst_name=None):
        src = Path(src)
        if not src.exists():
            print(f"[ARTIFACT] missing, skipped: {src}")
            return
        dst = artifact_dir / (dst_name or src.name)
        shutil.copy2(src, dst)
        print(f"[ARTIFACT] copied: {src} -> {dst}")

    
    copy_if_exists(best_ckpt, "best_tsn_3class.pth")
    copy_if_exists(cfg_path, "tsn_k400_3class.py")

    
    split_dir = WORK_ROOT / "mmaction2_custom_data" / "splits"
    for name in ["train_selected.csv", "val_selected.csv", "test_selected.csv"]:
        copy_if_exists(split_dir / name)

    
    for split, ann_path in ann_files.items():
        copy_if_exists(ann_path, f"{split}.txt")

    ann_root = WORK_ROOT / "mmaction2_custom_data" / "annotations"
    copy_if_exists(ann_root / "classes.txt")
    copy_if_exists(ann_root / "label_to_idx.json")

    
    for name in [
        "test_predictions_with_latency.csv",
        "mmaction2_latency_per_video.csv",
        "mmaction2_latency_summary.json",
        "test_metrics.json",
        "classification_report.txt",
        "research_details.json",
        "_summary.txt",
        "confusion_matrix.png",
        "confusion_matrix_norm.png",
    ]:
        copy_if_exists(work_dir / name)

    
    archive_base = WORK_ROOT / "fitness_paper_mmaction2_artifacts"
    archive_path = shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(artifact_dir)
    )

    print("\n" + "=" * 60)
    print("PERSISTENT ARTIFACT BUNDLE")
    print("=" * 60)
    print(f"Folder: {artifact_dir}")
    print(f"ZIP:    {archive_path}")
    print(
        "After the run finishes, use Kaggle 'Save Version' so these "
        "files become notebook outputs."
    )

    return artifact_dir, Path(archive_path)



def main():
    ensure_video_roots()

    print("[INFO] Installing MMAction2 + deps...")
    repo_dir = install_mmaction2()

    
    repo_str = str(repo_dir)
    if repo_str not in sys.path:
        sys.path.insert(0, repo_str)

    key_to_path_train = parse_video_keys(VIDEO_ROOTS["train"])
    key_to_path_val = parse_video_keys(VIDEO_ROOTS["val"])
    key_to_path_all = {**key_to_path_train, **key_to_path_val}
    print(f"[DATA] Parsed video keys: train={len(key_to_path_train)} "
          f"val={len(key_to_path_val)} union={len(key_to_path_all)}")

    print("[INFO] Loading CSVs and filtering to classes...")
    df_train = load_filtered_df(CSV_PATHS["train"], "train")
    df_val = load_filtered_df(CSV_PATHS["val"], "val")

    print("[INFO] Keeping only rows with existing mp4 files...")
    df_train = keep_only_existing(df_train, key_to_path_all, "train")
    df_val = keep_only_existing(df_val, key_to_path_all, "val")

    print("[INFO] Building splits (TEST first, then TRAIN/VAL, disjoint)...")
    df_train_sel, df_val_sel, df_test_sel = build_splits(df_train, df_val)

    split_dir = WORK_ROOT / "mmaction2_custom_data" / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)
    df_train_sel.to_csv(split_dir / "train_selected.csv", index=False)
    df_val_sel.to_csv(split_dir / "val_selected.csv", index=False)
    df_test_sel.to_csv(split_dir / "test_selected.csv", index=False)
    print(f"[SAVE] split CSVs -> {split_dir}")

    print("[INFO] Flattening videos + writing annotations...")
    ann_files, prefixes, label_to_idx = write_all_annotations_and_flatten(
        df_train_sel, df_val_sel, df_test_sel, key_to_path_all)
    print("[INFO] label_to_idx:", label_to_idx)

    print("[INFO] Writing custom TSN config...")
    cfg_path, work_dir = write_custom_config(repo_dir, ann_files, prefixes)

    print("[INFO] Training...")
    train_env = {"PYTHONPATH": str(repo_dir)}
    run([sys.executable, "tools/train.py", str(cfg_path),
         "--work-dir", str(work_dir)], cwd=str(repo_dir), extra_env=train_env)

    print("\n[DONE] Training finished.")

    best_ckpt = find_best_checkpoint(work_dir)
    print("[OUTPUT] Best/Latest checkpoint:", best_ckpt)

    extract_research_details(
        Path(cfg_path), work_dir,
        df_train_sel, df_val_sel, df_test_sel, best_ckpt)

    history = parse_mmengine_log_for_history(work_dir)
    plot_history(history, work_dir)

    test_ann = Path(ann_files["test"])
    test_prefix = Path(prefixes["test"])
    
    
    train_warmup_candidates = sorted(Path(prefixes["train"]).glob("*.mp4"))
    if not train_warmup_candidates:
        raise RuntimeError("No training video available for latency warm-up.")
    warmup_video = train_warmup_candidates[0]

    metrics = evaluate_on_test(
        Path(cfg_path), best_ckpt, test_ann, test_prefix, work_dir,
        n_warmup=10,
        warmup_video=warmup_video)

    summary_txt = work_dir / "_summary.txt"
    summary_lines = [
        "Experiment summary",
        f"Classes: {CLASSES}",
        f"Sampling: train={N_TRAIN_PER_CLASS}/class, "
        f"val={N_VAL_PER_CLASS}/class, "
        f"test={N_TEST_PER_CLASS}/class, seed={RANDOM_SEED}",
        f"Split sizes: train={len(df_train_sel)}, "
        f"val={len(df_val_sel)}, test={len(df_test_sel)}",
        f"Pretrained ckpt: {TSN_K400_CKPT}",
        f"Best checkpoint: {best_ckpt}",
        "",
        "Test metrics:",
        json.dumps(metrics, indent=2),
    ]
    summary_txt.write_text("\n".join(summary_lines), encoding="utf-8")
    print(f"[SAVE] {summary_txt}")

    
    
    save_persistent_artifacts(
        Path(cfg_path),
        work_dir,
        best_ckpt,
        ann_files,
    )


if __name__ == "__main__":
    main()
